# Zero-shot Classification (A/B): Plot Builder

Interactive plot builder for the zero-shot binary
classification experiments. Generates matplotlib previews
and copy-pasteable LaTeX (pgfplots) source.

## Experiment structure

Each sweep applies a **single perturbation type** to all
samples. The dropout sweeps set
`active_perturbations=[DROPOUT]` and `noise_std=0`;
the noise sweeps set `active_perturbations=[NOISE]` and
`dropout_rate=0`. So every run has **1000 samples of one
perturbation type**, with 0 baselines.

Each sweep contains **main** and **control** runs:

- **Main** (`dropout_noise`): the option labels are the
  actual perturbation names.
- **Masking/Jitter** (`masking_jitter`): the option labels
  are synonyms of the actual perturbation names.
- **Other controls**: arbitrary labels unrelated to the
  perturbations (`vanilla_chocolate`, `foo_bar`, etc.).

## Metrics

- **accuracy_primary**: fraction of correct picks (first
  token logit). $n = 1000$ per run. SE from Bernoulli.
- **logit_diff_correct**: logit(correct class) minus
  logit(wrong class), averaged over all $n = 1000$ samples
  in the run. SE/SD are empirical (within run). For the
  dropout panel this is dropout minus noise; for the noise
  panel, noise minus dropout.
- **agg_logit_diff_correct**: same using logsumexp
  (aggregate) logits instead of primary.

## Error bands

- **SE**: standard error of the mean.
- **SD**: standard deviation of individual outcomes.

## X axis

Perturbation strengths are shown as **percentiles**
(0 = weakest, 100 = strongest) so that the dropout and
noise panels are comparable despite different raw scales.

In [ ]:
import os
import pathlib

_this_dir = pathlib.Path(os.path.abspath("")).resolve()
if _this_dir.name == "paper":
    os.chdir(_this_dir.parent)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from wandb_cache import load_sweep

In [ ]:
PROJECT = "llm-mechanistic-detection"

# --- Sweep IDs (each contains main + control runs) ---
CLASS_DROPOUT_SWEEPS = [
    "rxrpp5xn",
    "tfhcmgrj",
    "wt49m58u",
    "m38olscs",  # original 4-model sweeps
    "8yzipesc",  # qwen3_32b extended aliases (dropout)
]
CLASS_NOISE_SWEEPS = [
    "ozwrrbx8",
    "sfn2calp",
    "e924abmp",
    "a72eirur",  # original 4-model sweeps
    "thhgb8rh",  # qwen3_32b extended aliases (noise)
]

# --- Columns ---
COL_MODEL = "model"
COL_DROPOUT = "perturbation.dropout_rate"
COL_NOISE = "perturbation.noise_std"
COL_ALIASES = "aliases"
MAIN_ALIAS = "dropout_noise"

# Aliases drawn with "main" style (solid, thick, markers)
MAIN_STYLE_ALIASES = {"dropout_noise", "masking_jitter"}

# --- Models (no Gemma) ---
MODELS = ["llama3_8b", "qwen3_14b", "qwen3_32b", "olmo3_32b"]

# --- Alias colors (from classification_curves_with_control) ---
# Named aliases get fixed colors; others get auto-assigned.
ALIAS_COLORS_MPL = {
    "dropout_noise": "#000000",
    "masking_jitter": "#c0392b",
    "rotation_permutation": "#1a5276",
    "scaling_translation": "#2e86c1",
    "vanilla_chocolate": "#1e7a1e",
    "foo_bar": "#5b2c8c",
    "x_y": "#8e44ad",
    "none": "#777777",
}
ALIAS_COLORS_LATEX = {
    "dropout_noise": "black",
    "masking_jitter": "red!70!black",
    "rotation_permutation": "blue!80!black",
    "scaling_translation": "blue!60",
    "vanilla_chocolate": "green!70!black",
    "foo_bar": "violet!80!black",
    "x_y": "violet!60",
    "none": "gray!70",
}
ALIAS_DISPLAY = {
    "dropout_noise": "Dropout/Noise",
    "masking_jitter": "Masking/Jitter",
    "rotation_permutation": "Rotation/Permutation",
    "scaling_translation": "Scaling/Translation",
    "vanilla_chocolate": "Vanilla/Chocolate",
    "foo_bar": "Foo/Bar",
    "x_y": "X/Y",
    "none": "None",
}

DROPOUT_XLABEL = {
    "mpl": "Perturbation percentile (dropout)",
    "latex": "Perturbation percentile (dropout)",
}
NOISE_XLABEL = {
    "mpl": "Perturbation percentile (noise)",
    "latex": "Perturbation percentile (noise)",
}

# --- Metrics ---
# All runs have 1000 samples of a single perturbation type.
# Accuracy: Bernoulli SE with n=1000.
# Logit diff: empirical SE from logged std/sqrt(1000).
METRICS = [
    "accuracy_primary",
    "accuracy_aggregate",
    "accuracy_argmax",
    "logit_diff_correct",
    "agg_logit_diff_correct",
]

In [ ]:
# --- Load data ---
def load_and_merge(sweep_ids, project):
    dfs = [load_sweep(sid, project=project) for sid in sweep_ids]
    return pd.concat(dfs, ignore_index=True)


print("Dropout sweeps:")
df_all_d = load_and_merge(CLASS_DROPOUT_SWEEPS, PROJECT)
print(f"  {len(df_all_d)} runs")

print("Noise sweeps:")
df_all_n = load_and_merge(CLASS_NOISE_SWEEPS, PROJECT)
print(f"  {len(df_all_n)} runs")

# Filter to target models
df_all_d = df_all_d[df_all_d[COL_MODEL].isin(MODELS)].copy()
df_all_n = df_all_n[df_all_n[COL_MODEL].isin(MODELS)].copy()

# --- Determine n per run ---
# Sweep configs set active_perturbations to a single type,
# so all 1000 stochastic samples are the same perturbation.
# num_baselines = 0 (sentences_file is set, no NOTHING class).
if "total_rows" in df_all_d.columns and "num_baselines" in df_all_d.columns:
    N_STOCHASTIC = int(
        (df_all_d["total_rows"] - df_all_d["num_baselines"]).mode().iloc[0]
    )
else:
    N_STOCHASTIC = 1000
print(f"\nSamples per run: {N_STOCHASTIC} (all same perturbation type)")

# --- Show per-model perturbation ranges ---
for name, df, x_col in [
    ("Dropout", df_all_d, COL_DROPOUT),
    ("Noise", df_all_n, COL_NOISE),
]:
    print(f"\n{name} ranges per model:")
    for model in MODELS:
        vals = sorted(df[df[COL_MODEL] == model][x_col].unique())
        if vals:
            print(f"  {model}: {vals[0]:.4f} .. {vals[-1]:.4f} ({len(vals)} points)")

# --- Preprocess: unified logit_diff columns ---
# Dropout sweep: all 1000 samples are DROPOUT.
# The logged "dropout_mean_logit_diff_dropout_vs_noise" is
# mean of (logit_dropout - logit_noise) over all 1000 samples.
# Positive means the model prefers the correct class.
_LD_D = "dropout_mean_logit_diff_dropout_vs_noise"
_LD_D_SE = "dropout_se_logit_diff_dropout_vs_noise"
_LD_D_SD = "dropout_std_logit_diff_dropout_vs_noise"
_ALD_D = "dropout_mean_aggregate_logit_diff_dropout_vs_noise"
_ALD_D_SE = "dropout_se_aggregate_logit_diff_dropout_vs_noise"
_ALD_D_SD = "dropout_std_aggregate_logit_diff_dropout_vs_noise"

df_all_d["logit_diff_correct"] = df_all_d[_LD_D]
df_all_d["logit_diff_correct_se"] = df_all_d[_LD_D_SE]
df_all_d["logit_diff_correct_sd"] = df_all_d[_LD_D_SD]
df_all_d["agg_logit_diff_correct"] = df_all_d[_ALD_D]
df_all_d["agg_logit_diff_correct_se"] = df_all_d[_ALD_D_SE]
df_all_d["agg_logit_diff_correct_sd"] = df_all_d[_ALD_D_SD]

# Noise sweep: all 1000 samples are NOISE.
# The logged column is still (dropout - noise), so we negate
# to get (noise - dropout) = (correct - incorrect).
_LD_N = "noise_mean_logit_diff_dropout_vs_noise"
_LD_N_SE = "noise_se_logit_diff_dropout_vs_noise"
_LD_N_SD = "noise_std_logit_diff_dropout_vs_noise"
_ALD_N = "noise_mean_aggregate_logit_diff_dropout_vs_noise"
_ALD_N_SE = "noise_se_aggregate_logit_diff_dropout_vs_noise"
_ALD_N_SD = "noise_std_aggregate_logit_diff_dropout_vs_noise"

df_all_n["logit_diff_correct"] = -df_all_n[_LD_N]  # negate
df_all_n["logit_diff_correct_se"] = df_all_n[_LD_N_SE]  # SE stays positive
df_all_n["logit_diff_correct_sd"] = df_all_n[_LD_N_SD]
df_all_n["agg_logit_diff_correct"] = -df_all_n[_ALD_N]
df_all_n["agg_logit_diff_correct_se"] = df_all_n[_ALD_N_SE]
df_all_n["agg_logit_diff_correct_sd"] = df_all_n[_ALD_N_SD]

print("\nLogit diff columns added (logit_diff_correct, agg_logit_diff_correct)")

# --- Discover aliases ---
ALL_ALIASES = sorted(df_all_d[COL_ALIASES].dropna().unique())
print(f"\nAliases ({len(ALL_ALIASES)}): {ALL_ALIASES}")

avail_metrics = [m for m in METRICS if m in df_all_d.columns]
print(f"Metrics: {avail_metrics}")

In [ ]:
# ── Helpers ──────────────────────────────────────────────────


def _alias_label(alias):
    return ALIAS_DISPLAY.get(alias, alias)


def _is_main_style(alias):
    return alias in MAIN_STYLE_ALIASES


def _per_model_pct(x_values):
    """Map a sorted array of x values to percentiles 0..100."""
    n = len(x_values)
    if n <= 1:
        return {x_values[0]: 50.0} if n == 1 else {}
    return {v: i * 100 / (n - 1) for i, v in enumerate(x_values)}


# ── Curve computation ────────────────────────────────────────


def compute_curves(df, x_col, metric, selected_aliases, n_stochastic, models):
    """Return {alias: {model: (x_pct, y, se, sd)}}.

    Each run has n_stochastic samples of a single perturbation.
    Percentiles are computed per model (each model has its own
    range of perturbation strengths from localization).

    - accuracy metrics: SE/SD from Bernoulli formula.
    - logit_diff metrics: SE/SD from precomputed columns.
    """
    is_pct = "accuracy" in metric or "f1" in metric
    is_ld = "logit_diff" in metric
    scale = 100 if is_pct else 1
    se_col = f"{metric}_se"
    sd_col = f"{metric}_sd"
    result = {}

    for alias in selected_aliases:
        sub_alias = df[df[COL_ALIASES] == alias]
        curves = {}
        for model in models:
            sub = sub_alias[sub_alias[COL_MODEL] == model]
            if metric not in sub.columns or sub.empty:
                continue

            grp = sub.groupby(x_col)
            avg = grp[metric].mean().sort_index() * scale
            if avg.empty:
                continue

            if is_pct:
                p = avg / 100
                pq = p * (1 - p)
                n_runs = grp[metric].count()
                n_total = n_runs * n_stochastic
                se = np.sqrt(pq / n_total) * 100
                sd = np.sqrt(pq) * 100
            elif is_ld and se_col in sub.columns:
                n_runs = grp[metric].count()
                if (n_runs > 1).any():
                    se = grp[metric].sem().sort_index()
                    sd = grp[sd_col].mean().sort_index()
                else:
                    se = grp[se_col].first().sort_index()
                    sd = grp[sd_col].first().sort_index()
            else:
                se = grp[metric].sem().sort_index() * scale
                sd = grp[metric].std().sort_index() * scale

            # Per-model percentile: this model's sorted
            # x-values map to 0, 10, 20, ..., 100
            x_raw = avg.index.values
            pct_map = _per_model_pct(x_raw)
            x_pct = np.array([pct_map[v] for v in x_raw])

            curves[model] = (x_pct, avg.values, se.values, sd.values)
        if curves:
            result[alias] = curves
    return result


# ── Matplotlib ───────────────────────────────────────────────


def plot_panel(ax, all_curves, x_label, metric, error_type, ymin, ymax, is_right=False):
    is_pct = "accuracy" in metric or "f1" in metric

    for alias, curves in all_curves.items():
        main_style = _is_main_style(alias)
        c = ALIAS_COLORS_MPL.get(alias, "#888888")
        ls = "-" if main_style else "--"
        lw = 3 if main_style else 1.5
        ms = 4 if main_style else 0
        alpha_band = 0.3 if main_style else 0.2
        zorder_line = 10 if main_style else 1

        for model in MODELS:
            if model not in curves:
                continue
            x, y, se, sd = curves[model]
            err = se if error_type == "SE" else sd
            label = _alias_label(alias) if model == MODELS[0] else None
            ax.plot(
                x,
                y,
                marker="o",
                markersize=ms,
                linewidth=lw,
                color=c,
                label=label,
                linestyle=ls,
                zorder=zorder_line,
            )
            ax.fill_between(
                x, y - err, y + err, alpha=alpha_band, color=c, zorder=zorder_line - 1
            )

    if is_pct:
        ax.axhline(50, color="gray", ls=":", alpha=0.5, label="chance")
    ax.set_xlabel(x_label)
    if is_right:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ylabel = (
            "Accuracy (%)"
            if is_pct
            else "Logit diff (correct \u2212 incorrect)"
            if "logit_diff" in metric
            else metric
        )
        ax.set_ylabel(ylabel)
    ax.set_xlim(-2, 102)
    ax.set_xticks(range(0, 101, 10))
    ax.grid(True, alpha=0.3)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


# ── LaTeX / pgfplots ────────────────────────────────────────


def latex_panel(
    all_curves,
    x_label,
    metric,
    error_type,
    models,
    is_right=False,
    ymin=None,
    ymax=None,
):
    is_pct = "accuracy" in metric or "f1" in metric
    is_ld = "logit_diff" in metric

    L = []
    L.append(r"\begin{tikzpicture}")
    L.append(r"\begin{axis}[")
    L.append("    height=7cm,")
    L.append("    width=\\linewidth,")
    L.append("    grid=major,")
    L.append(f"    xlabel={{{x_label}}},")
    if is_right:
        L.append("    ylabel={},")
        L.append("    yticklabels={},")
    else:
        if is_pct:
            L.append("    ylabel={Accuracy (\\%)},")
        elif is_ld:
            L.append("    ylabel={Logit diff (correct $-$ incorrect)},")
        else:
            L.append(f"    ylabel={{{metric}}},")
    L.append("    xmin=0, xmax=100,")
    L.append("    xtick={0,10,...,100},")
    if ymin is not None and ymax is not None:
        L.append(f"    ymin={ymin}, ymax={ymax},")
    L.append("    legend entries={},")
    L.append("]")

    for alias, curves in all_curves.items():
        main_style = _is_main_style(alias)
        c = ALIAS_COLORS_LATEX.get(alias, "gray")
        ls_tex = "" if main_style else "dashed, "
        lw_tex = "1.5pt" if main_style else "0.8pt"
        opacity = "0.3" if main_style else "0.15"

        for model in models:
            if model not in curves:
                continue
            x, y, se, sd = curves[model]
            err = se if error_type == "SE" else sd

            up = " ".join(f"({xi:.1f},{yi + ei:.4f})" for xi, yi, ei in zip(x, y, err))
            lo = " ".join(
                f"({xi:.1f},{yi - ei:.4f})"
                for xi, yi, ei in reversed(list(zip(x, y, err)))
            )
            L.append(
                f"\\addplot[{c}, fill={c}, "
                f"fill opacity={opacity}, "
                f"draw=none, forget plot] "
                f"coordinates {{{up} {lo}}} --cycle;"
            )
            coords = " ".join(f"({xi:.1f},{yi:.4f})" for xi, yi in zip(x, y))
            L.append(
                f"\\addplot[{c}, {ls_tex}mark=o, "
                f"mark size=1, line width={lw_tex}, "
                f"forget plot] "
                f"coordinates {{{coords}}};"
            )

    if is_pct:
        L.append(
            r"\addplot[gray, dashed, line width=0.5pt, "
            r"forget plot] coordinates {(0,50) (100,50)};"
        )
    L.append(r"\end{axis}")
    L.append(r"\end{tikzpicture}")
    return "\n".join(L)


def latex_figure(
    all_curves_d,
    all_curves_n,
    metric,
    error_type,
    models,
    selected_aliases,
    caption,
    label,
    ymin=None,
    ymax=None,
):
    d_tex = latex_panel(
        all_curves_d,
        DROPOUT_XLABEL["latex"],
        metric,
        error_type,
        models,
        ymin=ymin,
        ymax=ymax,
    )
    n_tex = latex_panel(
        all_curves_n,
        NOISE_XLABEL["latex"],
        metric,
        error_type,
        models,
        is_right=True,
        ymin=ymin,
        ymax=ymax,
    )

    items = []
    for a in selected_aliases:
        if a in all_curves_d or a in all_curves_n:
            c = ALIAS_COLORS_LATEX.get(a, "gray")
            al = _alias_label(a).replace("_", r"\_")
            ls = "thick" if _is_main_style(a) else "dashed"
            items.append(
                f"\\tikz\\draw[{c}, {ls}, mark=o, "
                f"mark size=1.5] plot coordinates "
                f"{{(0,0) (0.4,0)}}; {al}"
            )
    legend = "\\hspace{1em}".join(items)
    err_tag = "SE" if error_type == "SE" else "SD"

    return (
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"{legend}\n"
        "\\\\[6pt]\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {d_tex}\n"
        f"\\end{{minipage}}\n"
        "\\hfill\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {n_tex}\n"
        f"\\end{{minipage}}\n"
        f"\\caption{{{caption} "
        f"Bands show $\\pm${err_tag}.}}\n"
        f"\\label{{fig:{label}}}\n"
        "\\end{figure}"
    )

In [ ]:
# ── Interactive plot ──────────────────────────────────────────


# Auto-assign colors for aliases not in ALIAS_COLORS_MPL


_EXTRA_PALETTE = plt.cm.tab20.colors


_color_idx = 0


for a in ALL_ALIASES:
    if a not in ALIAS_COLORS_MPL:
        ALIAS_COLORS_MPL[a] = "#{:02x}{:02x}{:02x}".format(
            *[int(c * 255) for c in _EXTRA_PALETTE[_color_idx % 20][:3]]
        )

        _color_idx += 1


w_metric = widgets.Dropdown(
    options=avail_metrics, value=avail_metrics[0], description="Metric:"
)


w_error = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)


w_models = {
    m: widgets.Checkbox(value=True, description=m, indent=False) for m in MODELS
}


# Checkboxes for aliases (main always on, controls individually togglable)


w_alias_checks = {}


for a in ALL_ALIASES:
    w_alias_checks[a] = widgets.Checkbox(
        value=(a == MAIN_ALIAS),
        description=_alias_label(a),
        indent=False,
        layout=widgets.Layout(width="180px"),
    )


w_alias_btn_all = widgets.Button(
    description="Select all", layout=widgets.Layout(width="100px")
)


w_alias_btn_none = widgets.Button(
    description="Clear all", layout=widgets.Layout(width="100px")
)


w_alias_btn_main = widgets.Button(
    description="Main only", layout=widgets.Layout(width="100px")
)


def _on_alias_all(_):

    for cb in w_alias_checks.values():
        cb.value = True


def _on_alias_none(_):

    for cb in w_alias_checks.values():
        cb.value = False


def _on_alias_main(_):

    for cb in w_alias_checks.values():
        cb.value = False
    w_alias_checks[MAIN_ALIAS].value = True


w_alias_btn_all.on_click(_on_alias_all)
w_alias_btn_none.on_click(_on_alias_none)
w_alias_btn_main.on_click(_on_alias_main)

w_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
w_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
out = widgets.Output()


def redraw(*_):
    out.clear_output(wait=True)
    with out:
        metric = w_metric.value
        et = w_error.value
        models = [m for m in MODELS if w_models[m].value]
        sel = [a for a in ALL_ALIASES if w_alias_checks[a].value]
        ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
        ymax = w_ymax.value if not np.isnan(w_ymax.value) else None

        if not sel:
            print("No aliases selected.")
            return

        print(f"Selected: {sel}")

        cd = compute_curves(df_all_d, COL_DROPOUT, metric, sel, N_STOCHASTIC, models)
        cn = compute_curves(df_all_n, COL_NOISE, metric, sel, N_STOCHASTIC, models)

        if not cd and not cn:
            print("WARNING: no curves produced.")
            return

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        plot_panel(ax1, cd, DROPOUT_XLABEL["mpl"], metric, et, ymin, ymax)
        plot_panel(ax2, cn, NOISE_XLABEL["mpl"], metric, et, ymin, ymax, is_right=True)

        handles, labels = ax1.get_legend_handles_labels()
        fig.legend(
            handles=handles,
            loc="upper center",
            ncol=min(len(sel), 6),
            fontsize=9,
            bbox_to_anchor=(0.5, 1.08),
        )

        n_sel = len(sel)
        alias_summary = (
            "main only"
            if sel == [MAIN_ALIAS]
            else f"{n_sel} aliases"
            if n_sel > 2
            else " + ".join(_alias_label(a) for a in sel)
        )
        fig.suptitle(
            f"Zero-shot Classification ({alias_summary}, \u00b1{et})",
            fontsize=14,
            y=1.12,
        )
        fig.tight_layout()
        plt.show()

        caption = f"Zero-shot classification ({alias_summary})."
        label_tag = f"class-{n_sel}aliases"
        print(
            latex_figure(
                cd,
                cn,
                metric,
                et,
                models,
                sel,
                caption,
                label_tag,
                ymin=ymin,
                ymax=ymax,
            )
        )


for w in [w_metric, w_error, w_ymin, w_ymax]:
    w.observe(redraw, names="value")
for cb in w_models.values():
    cb.observe(redraw, names="value")
for cb in w_alias_checks.values():
    cb.observe(redraw, names="value")

display(
    widgets.HBox([w_metric, w_error, w_ymin, w_ymax]),
    widgets.HBox(list(w_models.values())),
    widgets.HBox([w_alias_btn_all, w_alias_btn_none, w_alias_btn_main]),
    widgets.GridBox(
        list(w_alias_checks.values()),
        layout=widgets.Layout(grid_template_columns="repeat(5, 180px)"),
    ),
    out,
)
redraw()